# Bab 8 — External Evaluation and Metric Comparison

Notebook pendamping Bab 8 Level 5.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Menjelaskan apa itu *external evaluation*, apa itu *ground truth*, dan kapan evaluasi eksternal mungkin dilakukan.
- Membedakan label hasil clustering dari label kelas sebenarnya, dan menjelaskan mengapa nomor kelompok bersifat sembarang (*cluster label permutation*).
- Membaca tabel kontingensi antara kelas dan kelompok.
- Menghitung dan menafsirkan *Adjusted Rand Index* (ARI), *Normalized Mutual Information* (NMI), *Homogeneity*, *Completeness*, dan *V-measure*.
- Menjelaskan mengapa metrik eksternal harus **tidak peka terhadap permutasi label**, dan mengapa tidak boleh memakai *accuracy* langsung.
- Menjelaskan keterbatasan evaluasi eksternal dan mengapa label acuan harus bermakna.
- Membandingkan metrik internal dan eksternal pada hasil empat algoritma Level 4, termasuk saat keduanya tidak sepakat.
- Memilih metrik evaluasi yang sesuai dengan situasi, dan bersiap untuk Level 6.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    completeness_score,
    davies_bouldin_score,
    homogeneity_score,
    normalized_mutual_info_score,
    silhouette_score,
    v_measure_score,
)
from sklearn.metrics.cluster import contingency_matrix
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

## Bagian B: Nomor kelompok itu sembarang

In [ ]:
# Label acuan (kelas sebenarnya) dan dua hasil clustering yang mengelompokkan data IDENTIK,
# hanya nomor kelompoknya yang berbeda
y_true = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
pred_same = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
pred_permuted = np.array([2, 2, 2, 0, 0, 0, 1, 1, 1])

print("Kecocokan langsung (pred == true), nomor sama    :", (pred_same == y_true).mean().round(3))
print("Kecocokan langsung (pred == true), nomor ditukar :", (pred_permuted == y_true).mean().round(3))
print("ARI nomor sama    :", adjusted_rand_score(y_true, pred_same))
print("ARI nomor ditukar :", adjusted_rand_score(y_true, pred_permuted))
print("NMI nomor ditukar :", normalized_mutual_info_score(y_true, pred_permuted))

## Bagian C: Tabel kontingensi

In [ ]:
# Hasil clustering yang tidak sempurna: dua titik salah kelompok
pred_mixed = np.array([0, 0, 1, 1, 1, 1, 2, 2, 0])
table = pd.crosstab(
    pd.Series(y_true, name="Kelas sebenarnya"),
    pd.Series(pred_mixed, name="Kelompok hasil clustering"),
)
print(table)

## Bagian D: ARI dihitung manual

In [ ]:
def ari_manual(labels_true, labels_pred):
    """Adjusted Rand Index dari tabel kontingensi, dengan menghitung pasangan titik."""
    cont = contingency_matrix(labels_true, labels_pred)
    n = cont.sum()
    sum_cells = sum(comb(int(v), 2) for v in cont.ravel())          # pasangan yang bersama di kelas DAN kelompok
    sum_rows = sum(comb(int(v), 2) for v in cont.sum(axis=1))       # pasangan yang bersama di kelas
    sum_cols = sum(comb(int(v), 2) for v in cont.sum(axis=0))       # pasangan yang bersama di kelompok
    expected = sum_rows * sum_cols / comb(int(n), 2)                # nilai yang diharapkan secara kebetulan
    maximum = 0.5 * (sum_rows + sum_cols)
    print(f"Pasangan sel = {sum_cells}, baris = {sum_rows}, kolom = {sum_cols}, "
          f"total pasangan = {comb(int(n), 2)}, diharapkan = {expected:.3f}, maksimum = {maximum:.1f}")
    return (sum_cells - expected) / (maximum - expected)


print(f"ARI manual        = {ari_manual(y_true, pred_mixed):.4f}")
print(f"ARI scikit-learn  = {adjusted_rand_score(y_true, pred_mixed):.4f}")

## Bagian E: Homogeneity, completeness, dan V-measure

In [ ]:
n_demo = len(y_true)
one_cluster = np.zeros(n_demo, dtype=int)                 # semua titik digabung jadi satu kelompok
each_alone = np.arange(n_demo)                            # setiap titik jadi kelompok sendiri

cases = {
    "Sempurna": pred_same,
    "Dua titik salah": pred_mixed,
    "Semua digabung (1 kelompok)": one_cluster,
    "Tiap titik sendiri (9 kelompok)": each_alone,
}
rows = []
for name, labels in cases.items():
    rows.append(
        {
            "Kasus": name,
            "Homogeneity": homogeneity_score(y_true, labels),
            "Completeness": completeness_score(y_true, labels),
            "V-measure": v_measure_score(y_true, labels),
            "NMI": normalized_mutual_info_score(y_true, labels),
            "ARI": adjusted_rand_score(y_true, labels),
        }
    )
print(pd.DataFrame(rows).set_index("Kasus").round(3).to_string())

## Bagian F: NMI dihitung manual

In [ ]:
def entropy(labels):
    """Entropi (dalam nat, logaritma natural) dari sebuah pelabelan."""
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -(p * np.log(p)).sum()


def nmi_manual(labels_true, labels_pred):
    """NMI = informasi bersama / rata-rata aritmetika dari dua entropi."""
    cont = contingency_matrix(labels_true, labels_pred).astype(float)
    n = cont.sum()
    pij = cont / n
    pi = pij.sum(axis=1, keepdims=True)
    pj = pij.sum(axis=0, keepdims=True)
    nonzero = pij > 0
    mutual_info = (pij[nonzero] * np.log(pij[nonzero] / (pi @ pj)[nonzero])).sum()
    h_true, h_pred = entropy(labels_true), entropy(labels_pred)
    print(f"H(kelas) = {h_true:.4f}, H(kelompok) = {h_pred:.4f}, informasi bersama = {mutual_info:.4f}")
    return mutual_info / (0.5 * (h_true + h_pred))


print(f"NMI manual        = {nmi_manual(y_true, pred_mixed):.4f}")
print(f"NMI scikit-learn  = {normalized_mutual_info_score(y_true, pred_mixed):.4f}")

## Bagian G: Tiga dataset dengan label acuan

In [ ]:
# Label acuan (y) hanya dipakai untuk MENILAI, tidak pernah diberikan ke algoritma clustering
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.5, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.06, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

datasets = {
    "Blobs": (StandardScaler().fit_transform(X_blobs), y_blobs, 3),
    "Moons": (StandardScaler().fit_transform(X_moons), y_moons, 2),
    "Circles": (StandardScaler().fit_transform(X_circles), y_circles, 2),
}
# eps DBSCAN dipilih setelah melihat data (belum pencarian parameter sistematis, itu materi Level 6)
dbscan_eps = {"Blobs": 0.30, "Moons": 0.30, "Circles": 0.30}
for name, (X, y, k) in datasets.items():
    print(f"{name:8s} bentuk data {X.shape}, jumlah kelas acuan = {k}, ukuran kelas = {np.bincount(y).tolist()}")

## Bagian H: Empat algoritma Level 4

In [ ]:
def run_algorithms(name, X, k):
    """Empat algoritma Level 4. Hanya X yang diberikan, tanpa label acuan."""
    return {
        "K-Means": KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X),
        "Hierarchical (Ward)": AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X),
        "DBSCAN": DBSCAN(eps=dbscan_eps[name], min_samples=5).fit_predict(X),
        "GMM": GaussianMixture(n_components=k, random_state=42).fit(X).predict(X),
    }


results = {name: run_algorithms(name, X, k) for name, (X, y, k) in datasets.items()}
for name, algos in results.items():
    dbscan_labels = algos["DBSCAN"]
    n_noise = int((dbscan_labels == -1).sum())
    n_found = len(set(dbscan_labels) - {-1})
    print(f"{name:8s} DBSCAN: {n_found} kelompok, {n_noise} titik noise dari {len(dbscan_labels)}")

## Bagian I: Tabel perbandingan metrik

In [ ]:
def internal_scores(X, labels):
    """Metrik internal. Noise DBSCAN (-1) DIKELUARKAN dulu, dan butuh minimal 2 kelompok tersisa."""
    mask = labels != -1
    Xm, lm = X[mask], labels[mask]
    if len(set(lm)) < 2:
        return {"Silhouette": np.nan, "Davies-Bouldin": np.nan, "Calinski-Harabasz": np.nan}
    return {
        "Silhouette": silhouette_score(Xm, lm),
        "Davies-Bouldin": davies_bouldin_score(Xm, lm),
        "Calinski-Harabasz": calinski_harabasz_score(Xm, lm),
    }


def external_scores(y, labels):
    """Metrik eksternal. Di sini noise (-1) DIPERLAKUKAN sebagai satu kelompok tersendiri."""
    return {
        "ARI": adjusted_rand_score(y, labels),
        "NMI": normalized_mutual_info_score(y, labels),
        "Homogeneity": homogeneity_score(y, labels),
        "Completeness": completeness_score(y, labels),
        "V-measure": v_measure_score(y, labels),
    }


all_tables = {}
for name, (X, y, k) in datasets.items():
    rows = []
    for algo, labels in results[name].items():
        row = {"Algoritma": algo, **internal_scores(X, labels), **external_scores(y, labels)}
        if algo == "K-Means":   # Inertia hanya bermakna untuk objektif centroid
            row["Inertia"] = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_
        rows.append(row)
    all_tables[name] = pd.DataFrame(rows).set_index("Algoritma")
    print(f"\n=== {name} ===")
    print("Metrik internal (butuh X dan label hasil clustering):")
    print(all_tables[name][["Silhouette", "Davies-Bouldin", "Calinski-Harabasz", "Inertia"]].round(3).to_string())
    print("Metrik eksternal (butuh label acuan):")
    print(all_tables[name][["ARI", "NMI", "Homogeneity", "Completeness", "V-measure"]].round(3).to_string())

## Bagian J: Kepekaan DBSCAN terhadap eps

In [ ]:
# Kepekaan DBSCAN terhadap eps pada Circles. PERINGATAN: label acuan dipakai di sini hanya
# untuk MENUNJUKKAN kepekaan. Dalam praktik, memilih eps agar ARI tinggi berarti memakai label
# untuk menyetel model (kebocoran informasi), sehingga skor eksternal tidak lagi jujur.
X_c, y_c, _ = datasets["Circles"]
for eps in (0.20, 0.25, 0.30, 0.35, 0.40):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_c)
    n_found = len(set(labels) - {-1})
    print(f"eps = {eps:.2f}: {n_found:2d} kelompok, {int((labels == -1).sum()):2d} noise, ARI = {adjusted_rand_score(y_c, labels):.3f}")

## Bagian K: Menggambar hasil tiap algoritma

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]
algo_names = list(results["Blobs"].keys())

fig, axes = plt.subplots(3, 4, figsize=(15, 10.5))
for row, (name, (X, y, k)) in enumerate(datasets.items()):
    for col, algo in enumerate(algo_names):
        labels = results[name][algo]
        ax = axes[row, col]
        for c in sorted(set(labels)):
            if c == -1:
                ax.scatter(X[labels == c, 0], X[labels == c, 1], s=22, marker="x", color="gray")
            else:
                ax.scatter(X[labels == c, 0], X[labels == c, 1], s=12, color=palette[c % len(palette)])
        ari = all_tables[name].loc[algo, "ARI"]
        ax.set_title(f"{name} - {algo}\nARI = {ari:.3f}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()
plt.show()

## Bagian L: Metrik internal dan eksternal berdampingan

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
width = 0.38
for ax, (name, table) in zip(axes, all_tables.items()):
    positions = np.arange(len(table))
    ax.bar(positions - width / 2, table["Silhouette"], width, color=palette[0], label="Silhouette (internal)")
    ax.bar(positions + width / 2, table["ARI"], width, color=palette[1], label="ARI (eksternal)")
    ax.set_xticks(positions)
    ax.set_xticklabels(table.index, rotation=20, fontsize=8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(name)
    ax.set_ylabel("Skor")
axes[1].legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## Bagian M: Mall Customers tanpa label acuan

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])

mall_results = {
    "K-Means": KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_mall),
    "Hierarchical (Ward)": AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(X_mall),
    "GMM": GaussianMixture(n_components=5, random_state=42).fit(X_mall).predict(X_mall),
}
mall_rows = [{"Algoritma": name, **internal_scores(X_mall, labels)} for name, labels in mall_results.items()]
print(pd.DataFrame(mall_rows).set_index("Algoritma").round(3).to_string())
print("Tidak ada label acuan pada Mall Customers, jadi metrik eksternal tidak bisa dihitung.")

## Bagian N: Label acuan harus bermakna

In [ ]:
# Gender adalah label yang ADA di tabel, tetapi apakah ia 'jawaban benar' untuk pengelompokan ini?
gender = (df["Gender"] == "Male").astype(int).to_numpy()
km_mall = mall_results["K-Means"]
print(f"ARI antara K-Means dan Gender : {adjusted_rand_score(gender, km_mall):.4f}")
print(f"NMI antara K-Means dan Gender : {normalized_mutual_info_score(gender, km_mall):.4f}")

## Latihan

1. Ubah `pred_mixed` menjadi `[0,0,0,1,1,1,2,2,1]` (hanya satu titik salah). Berapa ARI, NMI, Homogeneity, dan Completeness-nya?
2. Jalankan DBSCAN pada Moons dengan `eps` 0,2 dan 0,5. Berapa kelompok, noise, dan ARI-nya?
3. Pada Blobs, jalankan K-Means (K = 3, `random_state=42`). Hitung `(labels == y_blobs).mean()` dan bandingkan dengan ARI.
4. Jalankan K-Means dengan K = 2 pada Mall Customers dan hitung ARI terhadap `Gender`. Apa artinya?
5. Jelaskan mengapa pada Circles hasil DBSCAN (`eps` 0,30) punya Homogeneity 1,000 tetapi Completeness 0,892.
6. Mengapa kita tidak bisa menghitung ARI pada Mall Customers?
7. Mengapa DBSCAN pada Circles tampak terburuk menurut Silhouette tetapi terbaik menurut ARI?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Evaluasi eksternal** membandingkan hasil clustering dengan label acuan. Hanya mungkin bila label bermakna ada, dan label itu **tidak pernah** diberikan ke algoritma.
- Nomor kelompok **sembarang**: kecocokan langsung bernilai 0,0 pada pengelompokan yang sebenarnya sempurna. Gunakan metrik yang tidak peka permutasi: **ARI**, **NMI**, **Homogeneity**, **Completeness**, dan **V-measure**.
- **ARI** menghitung kecocokan pasangan titik dan dikoreksi terhadap kebetulan (0 = acak, 1 = sempurna). **NMI** berbasis informasi bersama tetapi tidak dikoreksi kebetulan. **Homogeneity** (kelompok murni) dan **Completeness** (kelas utuh) menunjukkan jenis kesalahan. V-measure adalah rata-rata harmonik keduanya.
- Perhitungan manual cocok dengan scikit-learn: ARI 0,3571 dan NMI 0,5895 pada sembilan titik.
- Pada eksperimen empat algoritma di tiga dataset, **metrik internal dan eksternal sering berlawanan**: DBSCAN sempurna pada Moons (ARI 1,000) tetapi terburuk menurut Silhouette (0,384), dan pada Circles ARI 0,961 dengan Silhouette 0,004.
- Perlakuan noise memengaruhi skor: pada Blobs, mengeluarkan 3 titik noise menaikkan skor internal DBSCAN sementara ARI-nya sedikit lebih rendah.
- DBSCAN sangat peka terhadap `eps`: 18 kelompok pada 0,20 dan 2 kelompok pada 0,35.
- Label acuan harus bermakna: ARI antara K-Means dan `Gender` pada Mall Customers hanya -0,0006.
- Tidak ada algoritma atau metrik yang terbaik di semua situasi. Gunakan beberapa metrik, gambar, dan pengetahuan domain.

### Persiapan untuk Level 6

Level 5 memberi kamu **alat ukur**. Level 6 (*Experimentation and Interpretation*) akan memakainya secara **sistematis**. Hal-hal yang akan dibahas di sana, dan sengaja tidak dilakukan di sini:

- **Desain eksperimen:** membandingkan algoritma dan parameter secara adil dan terencana.
- **Pencarian parameter:** menyetel `eps`, `K`, `min_samples`, dan lain-lain tanpa membocorkan label acuan.
- **Stabilitas:** mengulang dengan seed atau sampel berbeda, dan melihat apakah hasil bertahan.
- **Interpretasi:** menjelaskan arti tiap kelompok untuk keputusan nyata.
- **Pelaporan:** menyajikan hasil dengan jujur, termasuk ketidakpastian dan batasan metrik.

Bekal dari Level 5: arah tiap metrik, asumsinya, kapan ia menyesatkan, perlakuan noise, dan kebiasaan memakai lebih dari satu sudut pandang.